In [1]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm

from src.modules import ResynthesisModule, SpeakerModule
from anonymization.anonymizer import DemoAnonymizer

/data/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/mtseng/official_repo/DarkStream/notebooks/../src/components/losses.py:9: UserWarning: Module 'speechbrain.lobes.models.huggingface_transformers' was deprecated, redirecting to 'speechbrain.integrations.huggingface'. Please update your script.
  from speechbrain.lobes.models.huggingface_transformers.hubert import HuBERT
/data/mtseng/official_repo/DarkStream/notebooks/../anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Set the GPU device
MAX_WAV_VALUE = 32768.0

### Note
Use resynthesis_nola for 0ms lookahead, resynthesis_la7 for 140ms lookahead, and resynthesis_la14 for 280ms lookahead model

In [3]:
speaker_encoder = SpeakerModule(device="cuda")
resynthesizer = ResynthesisModule.load_from_checkpoint(
    "../ckpts/resynthesis_la14.ckpt",
    encoder_path=None,
    map_location="cuda"
)
resynthesizer.eval()

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
/data/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


ResynthesisModule(
  (encoder): PasrModule(
    (net): HiFiEncoder(
      (conv_pre): CausalConv(
        (layer): ParametrizedRawStreamingConv1d(
          1, 16, kernel_size=(7,), stride=(1,)
          (parametrizations): ModuleDict(
            (weight): ParametrizationList(
              (0): _WeightNorm()
            )
          )
        )
      )
      (downs): ModuleList(
        (0): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            16, 32, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (1): CausalConv(
          (layer): ParametrizedRawStreamingConv1d(
            32, 64, kernel_size=(4,), stride=(2,)
            (parametrizations): ModuleDict(
              (weight): ParametrizationList(
                (0): _WeightNorm()
              )
            )
          )
        )
        (2): 

In [4]:
wav_file = "../sample_wav/BDL/arctic_a0001.wav"
_device = "cuda"
sample_rate = 16000

audio, _ = librosa.load(wav_file, sr=sample_rate)

disp.Audio(audio, rate=16000)

### Non streaming synthesis

In [5]:
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))
resyth_audio = resynthesizer(audio, embed)
resyth_audio = resyth_audio.squeeze()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')

disp.Audio(resyth_audio, rate=16000)

### Streaming synthesis (One way)

In [6]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                _, z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:05<00:00, 29.22it/s]


### Streaming synthesis (Another way)

In [7]:
resynthesizer.encoder.net.streaming_forever(batch_size=1)
resynthesizer.decoder.streaming_forever(batch_size=1)
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    for offset in tqdm(range(0, original_length, frame_size)):
        frame = audio[:, :, offset:offset + frame_size]
        _, z_embed = resynthesizer.encoder.net(frame)
        if z_embed.shape[-1] == 0:
            continue
        resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
        all_audios.append(resynth_audio_chunk)   
        all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:05<00:00, 29.44it/s]


### Anonymization
For anonymization use embedding from anonymizer model instead

In [8]:
anonymizer = DemoAnonymizer(
    model_path=Path('../anonymization/models'), 
    model_tag='gan',
    device="cuda"
)

Model type of anonymizer: gan


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


In [9]:
anon_embed = anonymizer.anonymize_embedding(audio.squeeze().cpu().numpy(), 16000)
anon_embed = torch.cat([anon_embed[192:], anon_embed[:192]], dim=0) #embeds are concatenated in the reversed order

In [10]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                _, z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, anon_embed.unsqueeze(0))
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:05<00:00, 27.80it/s]
